# Tier 1: Pre-arXiv Replication and Strengthening

Runs all 4 Tier 1 checks in a single Colab session on A100.

| Check | What | Cost | Reviewer objection closed |
|-------|------|------|----------------------------|
| **T1.4** | 500 consecutive ablate-restore cycles on 1 head, checksum drift | ~1 min | 'Drift accumulates between 30-check intervals' |
| **T1.2** | L8H9 ablation effect across 5 eval datasets | ~5 min | 'L8H9 is wikitext-specific' |
| **T1.1** | Second-seed Type A replication across 8 checkpoints + alternate batch slice | ~30 min | 'Numbers are artifacts of specific seeds/batches' |
| **T1.3** | Bootstrap β CIs with 10,000 resamples | ~2 min (CPU) | 'Bootstrap CIs may be unstable at 2,000' |

**Runtime total:** ~40 minutes. **Cost:** ~$3 on Colab Pro A100.

**Results saved to Drive** `MyDrive/DFE_research/tier1/` plus auto-download to browser.

## Setup (mandatory Drive)

In [ ]:
!pip install -q transformers datasets torch accelerate scipy

In [ ]:
import torch, json, os, time, csv, hashlib, gc, urllib.request
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from scipy import stats as sp_stats

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print(f'GPU: {torch.cuda.get_device_name()}  |  Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    OUT_DIR = '/content/drive/MyDrive/DFE_research/tier1'
    os.makedirs(OUT_DIR, exist_ok=True)
    test = os.path.join(OUT_DIR, '.w')
    with open(test, 'w') as f: f.write('ok')
    os.remove(test)
    print(f'Drive mounted and verified; output to {OUT_DIR}')
except Exception as e:
    raise RuntimeError(f'Drive mount required: {e}')

GITHUB_RAW = 'https://raw.githubusercontent.com/mool32/functional-differentiation-dfe/main'

def log(msg):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True)

# Config matching Paper 2
MODEL_NAME = 'EleutherAI/pythia-410m-deduped'
N_LAYERS = 24
N_HEADS = 16
CHECKPOINTS = [512, 1000, 2000, 4000, 8000, 16000, 64000, 143000]
EVAL_N_BATCHES = 25
EVAL_BATCH_SIZE = 4
EVAL_SEQ_LEN = 2048
TYPE_A_ALPHA = 1.0
N_TYPE_A = 30
NEW_SEED_BASE = 12345  # Paper 2 used 42000

## Download Paper 2 ablation data (for T1.3)

In [ ]:
paper2_csv = os.path.join(OUT_DIR, 'paper2_all_ablations.csv')
if not os.path.exists(paper2_csv):
    log('Downloading Paper 2 ablation data...')
    urllib.request.urlretrieve(f'{GITHUB_RAW}/data/all_ablations.csv', paper2_csv)
log(f'Paper 2 data: {paper2_csv}')

## Prepare evaluation batches

Two slices of wikitext-103 train stream: original (Paper 2) and alternate (for T1.1 replication).

In [ ]:
from datasets import load_dataset
log('Loading tokenizer...')
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def stream_batches(dataset_spec, n_batches=EVAL_N_BATCHES, batch_size=EVAL_BATCH_SIZE,
                   seq_len=EVAL_SEQ_LEN, skip=0):
    """Stream a dataset and collect a specific slice of batches."""
    ds_name, ds_config, ds_split = dataset_spec
    try:
        ds = load_dataset(ds_name, ds_config, split=ds_split, streaming=True)
    except Exception as e:
        log(f'  Failed to load {ds_name}: {e}')
        return None
    need = n_batches * batch_size * seq_len
    skip_tokens = skip * batch_size * seq_len
    toks = []
    accumulated = 0
    for ex in ds:
        text_keys = ['text', 'content']
        text = None
        for k in text_keys:
            if k in ex:
                text = ex[k]
                break
        if not text or len(text.strip()) < 50: continue
        ids = tok(text, return_tensors='pt', truncation=False)['input_ids'].squeeze()
        if ids.dim() == 0: continue
        accumulated += ids.numel()
        if accumulated <= skip_tokens: continue
        toks.append(ids)
        if sum(t.numel() for t in toks) >= need * 1.2: break
    if not toks:
        return None
    merged = torch.cat(toks)[:need]
    if merged.numel() < need:
        log(f'  Short: got {merged.numel()}, need {need}')
        return None
    return merged.reshape(n_batches, batch_size, seq_len)

# Paper 2 original wikitext batches (for T1.1 baseline cross-check; comes from fresh stream start)
log('Preparing wikitext-103 original batch slice...')
wt_original = stream_batches(('wikitext', 'wikitext-103-raw-v1', 'train'), skip=0)
log(f'  Got {wt_original.shape if wt_original is not None else "None"}')

# Alternate wikitext-103 slice (different starting offset)
log('Preparing wikitext-103 ALTERNATE batch slice...')
wt_alternate = stream_batches(('wikitext', 'wikitext-103-raw-v1', 'train'), skip=200)
log(f'  Got {wt_alternate.shape if wt_alternate is not None else "None"}')

## Model operations

In [ ]:
def load_model(step):
    m = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, revision=f'step{step}', torch_dtype=torch.float32
    ).to(device).eval()
    return m

@torch.no_grad()
def evaluate_loss(model, batches):
    total = 0.0
    for i in range(batches.shape[0]):
        ids = batches[i].to(device)
        total += model(input_ids=ids, labels=ids).loss.item()
    return total / batches.shape[0]

def ablate_head(model, layer_idx, head_idx):
    head_dim = model.config.hidden_size // model.config.num_attention_heads
    s, e = head_idx * head_dim, (head_idx + 1) * head_dim
    w = model.gpt_neox.layers[layer_idx].attention.dense.weight
    saved = w.data.clone()
    w.data[:, s:e] = 0
    return saved

def restore_head(model, layer_idx, saved):
    model.gpt_neox.layers[layer_idx].attention.dense.weight.data.copy_(saved)

def perturb_type_a(model, block_idx, alpha, seed):
    g = torch.Generator(device=device).manual_seed(seed)
    block = model.gpt_neox.layers[block_idx]
    saved = {name: p.data.clone() for name, p in block.named_parameters()}
    for _, p in block.named_parameters():
        noise = torch.randn(p.shape, generator=g, device=device, dtype=p.dtype) * (p.data.std() * alpha)
        p.data.add_(noise)
    return saved

def restore_block(model, block_idx, saved):
    block = model.gpt_neox.layers[block_idx]
    for name, p in block.named_parameters():
        p.data.copy_(saved[name])

def tensor_hash(t):
    return hashlib.sha256(t.detach().cpu().contiguous().numpy().tobytes()).hexdigest()[:16]

def tensor_sumabs(t):
    return float(t.detach().abs().sum().item())

## Load step 143000 (used for T1.4, T1.2)

In [ ]:
log('Loading Pythia 410M step 143000 (float32)...')
model = load_model(143000)
log('Loaded.')

## T1.4 — Drift check on 500 consecutive ablate-restore cycles

Reviewer objection: 'drift accumulates between 30-ablation checks'.

Test: on a single head (L5H7), apply ablate→restore 500 times, measure checksum before/after each cycle. Verify drift <10⁻⁶ throughout.

In [ ]:
log('T1.4: 500 consecutive ablate-restore cycles on L5H7...')
L, H = 5, 7
w_ref = model.gpt_neox.layers[L].attention.dense.weight
initial_sum = tensor_sumabs(w_ref)
initial_hash = tensor_hash(w_ref.data)

drift_trajectory = []
hash_preserved_count = 0
max_drift = 0.0

t0 = time.time()
for i in range(500):
    s = ablate_head(model, L, H)
    restore_head(model, L, s)
    current_sum = tensor_sumabs(w_ref)
    drift = abs(current_sum - initial_sum)
    current_hash = tensor_hash(w_ref.data)
    drift_trajectory.append(drift)
    if current_hash == initial_hash:
        hash_preserved_count += 1
    if drift > max_drift:
        max_drift = drift
    if (i+1) % 100 == 0:
        log(f'  [{i+1}/500] drift={drift:.2e}  hash_match={current_hash==initial_hash}')

elapsed = time.time() - t0
log(f'T1.4 complete in {elapsed:.1f}s')
print(f'\nInitial sum-abs: {initial_sum:.6f}')
print(f'Max drift observed across 500 cycles: {max_drift:.2e}')
print(f'Hash preserved (bitwise identity): {hash_preserved_count}/500')
print(f'Final drift: {drift_trajectory[-1]:.2e}')

t14_result = {
    'n_cycles': 500,
    'initial_sum_abs': initial_sum,
    'max_drift': max_drift,
    'final_drift': drift_trajectory[-1],
    'hash_preserved_cycles': hash_preserved_count,
    'verdict': 'PASS' if max_drift < 1e-3 else 'FAIL',
}
print(f'\n>>> T1.4 verdict: {t14_result["verdict"]} <<<')

## T1.2 — L8H9 ablation effect across 5 eval datasets

Reviewer objection: 'L8H9 critical only for wikitext; not a real finding'.

Test: at step 143000, measure L8H9 ablation |Δ| on 5 datasets. Consistent |Δ|=0.1-0.2 would close objection.

In [ ]:
# Try to load 5 eval datasets (graceful fallback if any fail)
eval_specs = [
    ('wikitext-103', ('wikitext', 'wikitext-103-raw-v1', 'train')),
    ('pile-val', ('monology/pile-uncopyrighted', None, 'validation')),
    ('c4-en', ('allenai/c4', 'en', 'validation')),
    ('openwebtext', ('Skylion007/openwebtext', None, 'train')),
    ('bookcorpus', ('bookcorpus/bookcorpus', None, 'train')),
]

eval_batches = {}
for name, spec in eval_specs:
    log(f'Loading {name}...')
    try:
        b = stream_batches(spec)
        if b is not None:
            eval_batches[name] = b
            log(f'  OK  {b.shape}')
        else:
            log(f'  SKIP (empty)')
    except Exception as e:
        log(f'  SKIP ({e})')

log(f'Successfully loaded {len(eval_batches)} datasets')

In [ ]:
log('T1.2: L8H9 ablation on each dataset...')
t12_result = {'datasets': [], 'l8h9_delta_per_dataset': {}}

for name, batches in eval_batches.items():
    t0 = time.time()
    baseline = evaluate_loss(model, batches)
    s = ablate_head(model, 8, 9)
    ablated = evaluate_loss(model, batches)
    restore_head(model, 8, s)
    delta = -(ablated - baseline) / abs(baseline)
    elapsed = time.time() - t0
    
    t12_result['l8h9_delta_per_dataset'][name] = {
        'baseline_loss': baseline,
        'ablated_loss': ablated,
        'abs_delta': abs(delta),
        'delta': delta,
        'elapsed_sec': elapsed,
    }
    t12_result['datasets'].append(name)
    log(f'  {name:<15}: baseline={baseline:.4f}  |delta|={abs(delta):.4f}  ({elapsed:.1f}s)')

# Summary
deltas = [abs(t12_result['l8h9_delta_per_dataset'][d]['delta']) for d in t12_result['datasets']]
print(f'\nL8H9 |Δ| across {len(deltas)} datasets:')
print(f'  min: {min(deltas):.4f}')
print(f'  max: {max(deltas):.4f}')
print(f'  mean: {np.mean(deltas):.4f}')
print(f'  std: {np.std(deltas):.4f}')
print(f'  range/mean ratio: {(max(deltas)-min(deltas))/np.mean(deltas):.2f}')

t12_result['summary'] = {
    'min_abs_delta': min(deltas),
    'max_abs_delta': max(deltas),
    'mean_abs_delta': np.mean(deltas),
    'std_abs_delta': np.std(deltas),
    'n_datasets': len(deltas),
}
# Verdict: consistent if range is within 50% of mean
verdict = 'PASS' if (max(deltas) - min(deltas)) < 0.5 * np.mean(deltas) else 'INVESTIGATE'
t12_result['verdict'] = verdict
print(f'\n>>> T1.2 verdict: {verdict} <<<')

## T1.1 — Second-seed Type A replication + alternate batch

Reviewer objection: 'numbers are artifacts of specific seeds/batches'.

Test: rerun 30 Type A perturbations per checkpoint with seed base 12345 (vs 42000 in Paper 2). Use alternate wikitext batch slice. Compare resulting Δ statistics to Paper 2 Type A data.

In [ ]:
# Free current model, we'll reload per checkpoint
del model
gc.collect()
torch.cuda.empty_cache()

t11_result = {'checkpoints': {}}

for step in CHECKPOINTS:
    log(f'\n=== T1.1 checkpoint {step} ===')
    t0 = time.time()
    m = load_model(step)
    
    # Baseline on ALTERNATE batch slice (different from Paper 2)
    bl_alternate = evaluate_loss(m, wt_alternate)
    log(f'  baseline (alt batch): {bl_alternate:.4f}')
    
    # 30 Type A with NEW seed base
    deltas = []
    for i in range(N_TYPE_A):
        block_idx = i % N_LAYERS
        new_seed = NEW_SEED_BASE + i
        saved = perturb_type_a(m, block_idx, TYPE_A_ALPHA, new_seed)
        perturbed_loss = evaluate_loss(m, wt_alternate)
        restore_block(m, block_idx, saved)
        delta = -(perturbed_loss - bl_alternate) / abs(bl_alternate)
        deltas.append(delta)
    
    deltas = np.array(deltas)
    df_t, loc_t, sc_t = sp_stats.t.fit(deltas)
    aic_t = 2*3 - 2*np.sum(sp_stats.t.logpdf(deltas, df_t, loc_t, sc_t))
    mu, si = sp_stats.norm.fit(deltas)
    aic_n = 2*2 - 2*np.sum(sp_stats.norm.logpdf(deltas, mu, si))
    
    t11_result['checkpoints'][step] = {
        'baseline_loss': bl_alternate,
        'n_type_a': len(deltas),
        'mean': float(np.mean(deltas)),
        'std': float(np.std(deltas)),
        'skewness': float(sp_stats.skew(deltas)),
        'kurtosis': float(sp_stats.kurtosis(deltas)),
        'student_t_df': float(df_t),
        'delta_aic_t_vs_normal': float(aic_n - aic_t),
        'deltas': deltas.tolist(),
    }
    elapsed = time.time() - t0
    log(f'  std={t11_result["checkpoints"][step]["std"]:.5f}  '
        f'skew={t11_result["checkpoints"][step]["skewness"]:+.2f}  '
        f'dAIC(t-n)={aic_n-aic_t:+.1f}  '
        f't_df={df_t:.2f}  '
        f'({elapsed:.0f}s)')
    
    del m
    gc.collect()
    torch.cuda.empty_cache()

log('\nT1.1 complete.')

In [ ]:
# Compare with Paper 2 Type A results
import pandas as pd
paper2_df = pd.read_csv(paper2_csv)
paper2_typea = paper2_df[paper2_df['perturbation_type'] == 'type_a'].copy()

print('=== T1.1 COMPARISON: new seed (12345) + alternate batch vs Paper 2 original ===')
print(f'{"step":>6} | {"P2 std":>8} {"new std":>8} | {"P2 skew":>8} {"new skew":>8} | {"ratio std":>9}')
print('-' * 75)
ratios = []
for step in CHECKPOINTS:
    p2 = paper2_typea[paper2_typea['checkpoint'] == step]['delta'].values
    new_d = np.array(t11_result['checkpoints'][step]['deltas'])
    p2_std = p2.std()
    new_std = new_d.std()
    ratio = new_std / p2_std
    ratios.append(ratio)
    print(f'{step:>6} | {p2_std:>8.5f} {new_std:>8.5f} | {sp_stats.skew(p2):>+8.3f} {sp_stats.skew(new_d):>+8.3f} | {ratio:>9.3f}')

mean_ratio = np.mean(ratios)
max_deviation = max(abs(r - 1) for r in ratios)
print(f'\nMean std ratio (new/Paper 2): {mean_ratio:.3f}')
print(f'Max deviation from 1.0: {max_deviation:.3f} ({max_deviation*100:.1f}%)')

t11_result['comparison_to_paper2'] = {
    'mean_std_ratio': mean_ratio,
    'max_deviation': max_deviation,
    'per_checkpoint_ratios': dict(zip([str(s) for s in CHECKPOINTS], ratios)),
}

# Verdict: PASS if max deviation <10%, INVESTIGATE if 10-20%, FAIL if >20%
if max_deviation < 0.10:
    verdict = 'PASS'
elif max_deviation < 0.20:
    verdict = 'INVESTIGATE'
else:
    verdict = 'FAIL'
t11_result['verdict'] = verdict
print(f'\n>>> T1.1 verdict: {verdict} (threshold: <10% PASS, 10-20% INVESTIGATE, >20% FAIL) <<<')

## T1.3 — Bootstrap β CI with 10,000 resamples

Reviewer objection: 'CIs based on 2,000 resamples may be unstable'.

Test: recompute β bootstrap CI with 10,000 resamples on existing Paper 2 head deltas. Compare to original 2,000-resample CIs.

In [ ]:
paper2_heads = paper2_df[paper2_df['perturbation_type'] == 'head'].copy()

def bootstrap_beta(deltas, n_boot, seed=42, threshold=1e-4):
    rng = np.random.default_rng(seed)
    neg = -deltas[deltas < -threshold]
    if len(neg) < 5:
        return None, None, None
    bs = []
    for _ in range(n_boot):
        sample = rng.choice(neg, size=len(neg), replace=True)
        try:
            b, _, _ = sp_stats.gamma.fit(sample, floc=0)
            bs.append(b)
        except Exception:
            pass
    bs = np.array(bs)
    return float(np.median(bs)), float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))

log('T1.3: recomputing β CIs with 10,000 bootstrap resamples...')
t13_result = {'per_checkpoint': {}}

print(f'{"step":>6} | {"β (2k)":>20} | {"β (10k)":>20} | CI width ratio')
print('-' * 80)
for step in CHECKPOINTS:
    d = paper2_heads[paper2_heads['checkpoint']==step]['delta'].values
    med_2k, lo_2k, hi_2k = bootstrap_beta(d, n_boot=2000)
    med_10k, lo_10k, hi_10k = bootstrap_beta(d, n_boot=10000)
    w2k = (hi_2k - lo_2k) if (hi_2k is not None and lo_2k is not None) else None
    w10k = (hi_10k - lo_10k) if (hi_10k is not None and lo_10k is not None) else None
    width_ratio = w10k / w2k if (w2k and w10k) else None
    t13_result['per_checkpoint'][step] = {
        'beta_2k': {'median': med_2k, 'lo': lo_2k, 'hi': hi_2k},
        'beta_10k': {'median': med_10k, 'lo': lo_10k, 'hi': hi_10k},
        'ci_width_ratio_10k_over_2k': width_ratio,
    }
    if med_2k is None:
        print(f'{step:>6} | {"(too few)":>20} | {"(too few)":>20} | -')
    else:
        s2k = f'{med_2k:.3f} [{lo_2k:.2f},{hi_2k:.2f}]'
        s10k = f'{med_10k:.3f} [{lo_10k:.2f},{hi_10k:.2f}]'
        print(f'{step:>6} | {s2k:>20} | {s10k:>20} | {width_ratio:.3f}')

# Median point estimates should be nearly identical; CI width should tighten slightly
valid = [s for s in CHECKPOINTS if t13_result['per_checkpoint'][s]['beta_2k']['median'] is not None]
med_diffs = [abs(t13_result['per_checkpoint'][s]['beta_10k']['median'] - 
             t13_result['per_checkpoint'][s]['beta_2k']['median']) for s in valid]
max_med_diff = max(med_diffs) if med_diffs else 0
t13_result['max_median_drift_2k_to_10k'] = max_med_diff

verdict = 'PASS' if max_med_diff < 0.05 else 'INVESTIGATE'
t13_result['verdict'] = verdict
print(f'\nMax median-β drift between 2k and 10k bootstrap: {max_med_diff:.3f}')
print(f'>>> T1.3 verdict: {verdict} <<<')

## Save all results + force-download

In [ ]:
tier1_summary = {
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'model': MODEL_NAME,
    'T1_1_second_seed_replication': t11_result,
    'T1_2_l8h9_cross_dataset': t12_result,
    'T1_3_bootstrap_10k': t13_result,
    'T1_4_drift_500_cycles': t14_result,
    'overall_verdict': {
        'T1.1': t11_result['verdict'],
        'T1.2': t12_result['verdict'],
        'T1.3': t13_result['verdict'],
        'T1.4': t14_result['verdict'],
    },
}

summary_path = os.path.join(OUT_DIR, 'tier1_summary.json')
with open(summary_path, 'w') as f:
    json.dump(tier1_summary, f, indent=2, default=str)

log('=== TIER 1 OVERALL SUMMARY ===')
for k, v in tier1_summary['overall_verdict'].items():
    print(f'  {k}: {v}')

all_pass = all(v == 'PASS' for v in tier1_summary['overall_verdict'].values())
print(f'\n>>> All T1 checks PASS: {all_pass} <<<')
print(f'\nSaved to {summary_path}')

# Force browser download
try:
    from google.colab import files
    files.download(summary_path)
    print('Download triggered.')
except Exception as e:
    print(f'Manual download from Drive: {OUT_DIR}/tier1_summary.json')